In [ ]:
!pip install rioxarray xarray netCDF4 requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.0 MB/s eta 0:00:00


In [ ]:
import os
import requests
import pandas as pd
import xarray as xr
import rioxarray as rxr

BASE_URL = "https://data.chc.ucsb.edu/products/CHIRPS/v3.0/monthly/latam/tifs/"
DOWNLOAD_DIR = "/content/chirps_tifs"      # temporary storage for .tif files
OUTPUT_DIR = "/content/chirps_netcdf"      # final .nc files land here

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## functions to download, join and save as NetCDF

In [ ]:
def download_file(url, dest_path):
    """Download a single file if it doesn't already exist locally."""
    if os.path.exists(dest_path):
        return True
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except requests.exceptions.HTTPError:
        # e.g. month not yet published (common for the most recent year)
        print(f"  Not available: {os.path.basename(url)}")
        return False
    except requests.exceptions.RequestException as e:
        print(f"  Failed to download {url}: {e}")
        return False

In [ ]:
def build_year_dataset(year, delete_tifs_after=True , fill_value=-9999.0):
    """
    Downloads the 12 monthly GeoTIFFs for a given year, joins them along
    a new 'time' dimension using Xarray, and saves the result as NetCDF4.
    """
    print(f"Processing year {year}...")
    monthly_das = []

    for month in range(1, 13):
        mm = f"{month:02d}"
        filename = f"chirps-v3.0.{year}.{mm}.tif"
        url = BASE_URL + filename
        local_path = os.path.join(DOWNLOAD_DIR, filename)

        if not download_file(url, local_path):
            continue  # skip months that don't exist yet (e.g. late 2026)

        da = rxr.open_rasterio(local_path, masked=True)
        da = da.squeeze("band", drop=True)  # GeoTIFFs load with a 'band' dim we don't need
        da = da.expand_dims(time=[pd.Timestamp(year=year, month=month, day=1)])
        da.name = "precip"
        monthly_das.append(da)

    if not monthly_das:
        print(f"  No files found for {year}, skipping.\n")
        return None

    # Join the monthly layers along time -> single 3D (time, y, x) array
    year_da = xr.concat(monthly_das, dim="time")
    year_da = year_da.rio.write_crs("EPSG:4326", inplace=False) # Keep the CRS attached to the array
    year_ds = year_da.to_dataset()

    # Rename to CF-standard coordinate names R can auto-recognize
    year_ds = year_ds.rename({"x": "lon", "y": "lat"})
    year_ds["lon"].attrs = {"standard_name": "longitude", "long_name": "longitude",
                             "units": "degrees_east", "axis": "X"}
    year_ds["lat"].attrs = {"standard_name": "latitude", "long_name": "latitude",
                             "units": "degrees_north", "axis": "Y"}

    year_ds.attrs["source"] = "CHIRPS v3.0 monthly, LatAm subset"
    year_ds["precip"].attrs["long_name"] = "monthly precipitation"
    year_ds["precip"].attrs["units"] = "mm/month"

    # Replace NaN with an explicit numeric nodata value, and set it as missing_value too (some R readers look for that attribute instead)
    year_ds["precip"] = year_ds["precip"].fillna(fill_value)
    year_ds["precip"].attrs["missing_value"] = fill_value
    encoding = {
        "precip": {
            "_FillValue": fill_value,   # <- the key fix: numeric, not NaN
            "dtype": "float32",
            "zlib": True,
            "complevel": 4,
        }
    }

    out_path = os.path.join(OUTPUT_DIR, f"chirps-v3.0.{year}.monthly.nc")
    year_ds.to_netcdf(out_path, engine="netcdf4", encoding=encoding, format="NETCDF4")
    print(f"  Saved: {out_path} ({year_ds.sizes['time']} months)\n")

    if delete_tifs_after:
        for month in range(1, 13):
            mm = f"{month:02d}"
            fp = os.path.join(DOWNLOAD_DIR, f"chirps-v3.0.{year}.{mm}.tif")
            if os.path.exists(fp):
                os.remove(fp)

    return year_ds

## download individual years

In [ ]:
ds_1981 = build_year_dataset(1981)

Processing year 1981...
  Saved: /content/chirps_netcdf/chirps-v3.0.1981.monthly.nc (12 months)



In [ ]:
ds_y = build_year_dataset(1983)

Processing year 1983...
  Saved: /content/chirps_netcdf/chirps-v3.0.1983.monthly.nc (12 months)



In [ ]:
for year in range(1984, 2026):  # 2026 will only have Jan–Jul
    build_year_dataset(year)

Processing year 1984...
  Saved: /content/chirps_netcdf/chirps-v3.0.1984.monthly.nc (12 months)

Processing year 1985...
  Saved: /content/chirps_netcdf/chirps-v3.0.1985.monthly.nc (12 months)

Processing year 1986...
  Saved: /content/chirps_netcdf/chirps-v3.0.1986.monthly.nc (12 months)

Processing year 1987...
  Saved: /content/chirps_netcdf/chirps-v3.0.1987.monthly.nc (12 months)

Processing year 1988...
  Saved: /content/chirps_netcdf/chirps-v3.0.1988.monthly.nc (12 months)

Processing year 1989...
  Saved: /content/chirps_netcdf/chirps-v3.0.1989.monthly.nc (12 months)

Processing year 1990...
  Saved: /content/chirps_netcdf/chirps-v3.0.1990.monthly.nc (12 months)

Processing year 1991...
  Saved: /content/chirps_netcdf/chirps-v3.0.1991.monthly.nc (12 months)

Processing year 1992...
  Saved: /content/chirps_netcdf/chirps-v3.0.1992.monthly.nc (12 months)

Processing year 1993...
  Saved: /content/chirps_netcdf/chirps-v3.0.1993.monthly.nc (12 months)

Processing year 1994...
  Save

In [ ]:
ds_y = build_year_dataset(2026)

Processing year 2026...
  Not available: chirps-v3.0.2026.08.tif
  Not available: chirps-v3.0.2026.09.tif
  Not available: chirps-v3.0.2026.10.tif
  Not available: chirps-v3.0.2026.11.tif
  Not available: chirps-v3.0.2026.12.tif
  Saved: /content/chirps_netcdf/chirps-v3.0.2026.monthly.nc (7 months)



## join yearly file into one file

In [ ]:
import glob
import xarray as xr

# Find all yearly files, sorted so time stays in order
nc_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "chirps-v3.0.*.monthly.nc")))
print(f"Found {len(nc_files)} yearly files")

# Open and concatenate them all along the time dimension.
# chunks='auto' uses dask so this doesn't try to load everything into RAM at once.
combined_ds = xr.open_mfdataset(
    nc_files,
    combine="by_coords",
    chunks={"time": 12},
    engine="netcdf4",
)

# Make sure time is sorted (glob sort should already guarantee this, but it's a cheap safety check)
combined_ds = combined_ds.sortby("time")

print(combined_ds)

fill_value = -9999.0

combined_ds["precip"] = combined_ds["precip"].fillna(fill_value)
combined_ds["precip"].attrs["missing_value"] = fill_value
combined_ds["precip"].attrs["long_name"] = "monthly precipitation"
combined_ds["precip"].attrs["units"] = "mm/month"

encoding = {
    "precip": {
        "_FillValue": fill_value,
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
    },
    "time": {"dtype": "float64"},  # keeps time encoding numeric/consistent on write
}

out_path = os.path.join(OUTPUT_DIR, "chirps-v3.0.1981-2026.monthly.nc")
combined_ds.to_netcdf(out_path, engine="netcdf4", encoding=encoding, format="NETCDF4")
print(f"Saved combined file: {out_path}")

Found 46 yearly files
<xarray.Dataset> Size: 14GB
Dimensions:      (time: 547, lat: 1900, lon: 1720)
Coordinates:
  * time         (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-07-01
  * lat          (lat) float64 15kB 34.97 34.92 34.87 ... -59.88 -59.93 -59.98
  * lon          (lon) float64 14kB -120.0 -119.9 -119.9 ... -34.07 -34.02
    spatial_ref  int64 8B 0
Data variables:
    precip       (time, lat, lon) float64 14GB dask.array<chunksize=(12, 634, 574), meta=np.ndarray>
Attributes:
    source:   CHIRPS v3.0 monthly, LatAm subset
Saved combined file: /content/chirps_netcdf/chirps-v3.0.1981-2026.monthly.nc


## crop the NetCDF to a smaller area

In [ ]:
# load the LATAM NetCDF
import xarray as xr

ds = xr.open_dataset('/content/chirps_netcdf/chirps-v3.0.1981-2026.monthly.nc')

# Bounding box
lon_min, lon_max = -82, -66
lat_min, lat_max = -5, 13

# CHIRPS tifs are usually stored with lat DESCENDING (north -> south), so the slice direction for lat depends on how it's ordered in your file.
if ds.lat.values[0] > ds.lat.values[-1]:
    lat_slice = slice(lat_max, lat_min)
else:
    lat_slice = slice(lat_min, lat_max)

# Crop the dataset
ds_col = ds.sel(lon = slice(lon_min,lon_max),
                  lat = lat_slice)

print(ds_col)

<xarray.Dataset> Size: 252MB
Dimensions:      (time: 547, lat: 360, lon: 320)
Coordinates:
  * time         (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-07-01
  * lat          (lat) float64 3kB 12.97 12.92 12.87 ... -4.875 -4.925 -4.975
  * lon          (lon) float64 3kB -81.97 -81.92 -81.87 ... -66.12 -66.07 -66.02
    spatial_ref  int64 8B ...
Data variables:
    precip       (time, lat, lon) float32 252MB ...
Attributes:
    source:   CHIRPS v3.0 monthly, LatAm subset


In [ ]:
fill_value = -9999.0

ds_col["precip"] = ds_col["precip"].fillna(fill_value)
ds_col["precip"].attrs["missing_value"] = fill_value
ds_col["precip"].attrs["long_name"] = "monthly precipitation"
ds_col["precip"].attrs["units"] = "mm/month"

encoding = {
    "precip": {
        "_FillValue": fill_value,
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
    },
    "time": {"dtype": "float64"},
}

OUTPUT_DIR = "/content/chirps_netcdf"
out_path = os.path.join(OUTPUT_DIR, "chirps-v3.0.1981-2026.colombia.nc")
ds_col.to_netcdf(out_path, engine="netcdf4", encoding=encoding, format="NETCDF4")
print(f"Saved cropped file: {out_path}")

Saved cropped file: /content/chirps_netcdf/chirps-v3.0.1981-2026.colombia.nc


# download newly available month

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray as rxr

BASE_URL = "https://data.chc.ucsb.edu/products/CHIRPS/v3.0/monthly/latam/tifs/"
DOWNLOAD_DIR = "/content/chirps_tifs"
OUTPUT_DIR = "/content/chirps_netcdf"

year, month = 2026, 8

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def download_file(url, dest_path):
    if os.path.exists(dest_path):
        return True
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except requests.exceptions.HTTPError:
        print(f"  Not available: {os.path.basename(url)}")
        return False

In [ ]:
mm = f"{month:02d}"
filename = f"chirps-v3.0.{year}.{mm}.tif"
url = BASE_URL + filename
local_tif_path = os.path.join(DOWNLOAD_DIR, filename)

ok = download_file(url, local_tif_path)
if not ok:
    raise RuntimeError(f"{filename} is not available yet on the server.")

da = rxr.open_rasterio(local_tif_path, masked=True)
da = da.squeeze("band", drop=True)
da = da.expand_dims(time=[pd.Timestamp(year=year, month=month, day=1)])
da.name = "precip"

# Match the CF-compliant format used for all your other files, so it
# concatenates cleanly with the historical dataset later
da = da.rio.write_crs("EPSG:4326", inplace=False)
ds_aug = da.to_dataset()
ds_aug = ds_aug.rename({"x": "lon", "y": "lat"})
ds_aug["lon"].attrs = {"standard_name": "longitude", "long_name": "longitude",
                        "units": "degrees_east", "axis": "X"}
ds_aug["lat"].attrs = {"standard_name": "latitude", "long_name": "latitude",
                        "units": "degrees_north", "axis": "Y"}
ds_aug.attrs["source"] = "CHIRPS v3.0 monthly, LatAm subset"
ds_aug["precip"].attrs["long_name"] = "monthly precipitation"
ds_aug["precip"].attrs["units"] = "mm"

fill_value = -9999.0
ds_aug["precip"] = ds_aug["precip"].fillna(fill_value)
ds_aug["precip"].attrs["missing_value"] = fill_value

encoding = {
    "precip": {"_FillValue": fill_value, "dtype": "float32", "zlib": True, "complevel": 4}
}

aug_out_path = os.path.join(OUTPUT_DIR, f"chirps-v3.0.{month}-{year}.monthly.nc")
ds_aug.to_netcdf(aug_out_path, engine="netcdf4", encoding=encoding, format="NETCDF4")
print(f"Saved: {aug_out_path}")

os.remove(local_tif_path)  # cleanup, matches your earlier pattern

Saved: /content/chirps_netcdf/chirps-v3.0.august-2026.monthly.nc


Crop to colombia

In [ ]:
# load the LATAM NetCDF
import xarray as xr

ds = xr.open_dataset(f'/content/chirps_netcdf/chirps-v3.0.{month}-{year}.monthly.nc')

# Bounding box
lon_min, lon_max = -82, -66
lat_min, lat_max = -5, 13

# CHIRPS tifs are usually stored with lat DESCENDING (north -> south), so the slice direction for lat depends on how it's ordered in your file.
if ds.lat.values[0] > ds.lat.values[-1]:
    lat_slice = slice(lat_max, lat_min)
else:
    lat_slice = slice(lat_min, lat_max)

# Crop the dataset
ds_col = ds.sel(lon = slice(lon_min,lon_max),
                  lat = lat_slice)

print(ds_col)

<xarray.Dataset> Size: 927kB
Dimensions:      (time: 1, lat: 360, lon: 320)
Coordinates:
  * time         (time) datetime64[ns] 8B 2026-08-01
  * lat          (lat) float64 3kB 12.97 12.92 12.87 ... -4.875 -4.925 -4.975
  * lon          (lon) float64 3kB -81.97 -81.92 -81.87 ... -66.12 -66.07 -66.02
    spatial_ref  int64 8B ...
Data variables:
    precip       (time, lat, lon) float64 922kB ...
Attributes:
    source:   CHIRPS v3.0 monthly, LatAm subset


load the 1981- 2026.XX month

In [ ]:
ds_hist = xr.open_dataset(f'/content/chirps_netcdf/chirps-v3.0.1981-{year}-{month -1}.colombia.nc')

# Crop it to the same Colombia bounding box, so both datasets share identical
# lat/lon grids before concatenating. If this file is already the Colombia-cropped
# version, this selection is harmless (it will simply return the same domain).
if ds_hist.lat.values[0] > ds_hist.lat.values[-1]:
    lat_slice_hist = slice(lat_max, lat_min)
else:
    lat_slice_hist = slice(lat_min, lat_max)

ds_hist_col = ds_hist.sel(lon=slice(lon_min, lon_max), lat=lat_slice_hist)

print(ds_hist_col)

<xarray.Dataset> Size: 252MB
Dimensions:      (time: 547, lat: 360, lon: 320)
Coordinates:
  * time         (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-07-01
  * lat          (lat) float64 3kB 12.97 12.92 12.87 ... -4.875 -4.925 -4.975
  * lon          (lon) float64 3kB -81.97 -81.92 -81.87 ... -66.12 -66.07 -66.02
    spatial_ref  int64 8B ...
Data variables:
    precip       (time, lat, lon) float32 252MB ...
Attributes:
    source:   CHIRPS v3.0 monthly, LatAm subset


Join both datasets and save

In [ ]:
ds_combined = xr.concat([ds_hist_col, ds_col], dim="time")
ds_combined = ds_combined.sortby("time")

# Safety check: no duplicate months after the join
_, unique_idx = np.unique(ds_combined["time"].values, return_index=True)
if len(unique_idx) != ds_combined.sizes["time"]:
    print("Warning: duplicate time steps found, keeping only the first occurrence.")
    ds_combined = ds_combined.isel(time=sorted(unique_idx))

print(ds_combined)
print("Time range:", ds_combined.time.min().values, "to", ds_combined.time.max().values)

# Save with the same fill-value/CF encoding used throughout
fill_value = -9999.0
ds_combined["precip"] = ds_combined["precip"].fillna(fill_value)
ds_combined["precip"].attrs["missing_value"] = fill_value

encoding = {
    "precip": {"_FillValue": fill_value, "dtype": "float32", "zlib": True, "complevel": 4},
    "time": {"dtype": "float64"},
}

combined_out_path = f"/content/chirps_netcdf/chirps-v3.0.1981-{year}-{month}.monthly.colombia.nc"
ds_combined.to_netcdf(combined_out_path, engine="netcdf4", encoding=encoding, format="NETCDF4")
print(f"Saved: {combined_out_path}")

<xarray.Dataset> Size: 505MB
Dimensions:      (time: 548, lat: 360, lon: 320)
Coordinates:
  * time         (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-08-01
  * lat          (lat) float64 3kB 12.97 12.92 12.87 ... -4.875 -4.925 -4.975
  * lon          (lon) float64 3kB -81.97 -81.92 -81.87 ... -66.12 -66.07 -66.02
    spatial_ref  int64 8B 0
Data variables:
    precip       (time, lat, lon) float64 505MB nan nan nan ... 31.03 30.83
Attributes:
    source:   CHIRPS v3.0 monthly, LatAm subset
Time range: 1981-01-01T00:00:00.000000000 to 2026-08-01T00:00:00.000000000
Saved: /content/chirps_netcdf/chirps-v3.0.1981-2026-08.monthly.colombia.nc
